# 프롬프트 Template을 위한 Synthetic Test Data 생성

대략 다음과 같은 프롬프트가 있다고 상상해 보세요.

"""Here's some things I want you to analyze:

<thing>
{{thing1}}
</thing>
<thing>
{{thing2}}
</thing>

These things are [description of things]. Please read them carefully and [do some task]."""

여기서 thing1과 thing2를 "변수"라고 부르며, 가능한 다양한 thing1과 thing2 값에서 프롬프트가 잘 동작하기를 원합니다.

이 프롬프트 template을 어떻게 테스트할 수 있을까요? 넣어 볼 실제 값이 있을 수도 있습니다. 하지만 없을 수도 있고, privacy 때문에 가지고 있는 값으로 테스트할 수 없을 수도 있습니다. 걱정하지 마세요. Claude가 만들어 낼 수 있습니다! 이 cookbook은 Claude 및 Claude API를 사용해 프롬프트용 synthetic test data를 생성하는 방법을 보여 줍니다. template에서 변수를 추출하고, example block을 구성하고, test case를 생성하고, 결과를 반복적으로 개선하는 함수가 포함되어 있습니다. 이점은 두 가지입니다.

1. 프롬프트 평가
이 test case로 Claude가 현실적인 예제에서 어떻게 수행할지 확인할 수 있습니다.

2. Multishot 예제로 프롬프트 개선
Claude에 예제를 제공하는 것은 성능 개선을 위한 가장 좋은 방법일 수 있습니다. 이 notebook은 이상적인 input/output 쌍을 얻는 데 절반에 해당하는 현실적인 입력을 생성하는 데 도움이 됩니다.

In [ ]:
% pip install anthropic IPython

In [1]:
import re

import anthropic

# 여기에 API key 입력
api_key = ""
CLIENT = anthropic.Anthropic(api_key=api_key)
MODEL_NAME = "claude-sonnet-4-6"

먼저 이 notebook 전체에서 사용할 helper 함수를 정의합니다.

In [2]:
# 먼저 `extract_variables` 함수,
# 프롬프트 template을 받아 이중 중괄호 안에 있는 "변수"를 추출합니다.
def extract_variables(prompt_template):
    """프롬프트 template에서 변수를 추출합니다."""
    pattern = r"{{([^}]+)}}"
    variables = re.findall(pattern, prompt_template)
    return set(variables)


# 다음은 줄바꿈으로 연결하기만 하는 `construct_variables_names`입니다.
def construct_variables_names(prompt_template):
    """프롬프트 template에서 변수 이름 문자열을 구성합니다."""
    variables = extract_variables(prompt_template)
    return "\n".join(variables)


# `construct_variables_block` 함수는 변수 목록을 받아 "variables block"을 만듭니다.
# 변수가 'animal'과 'topic'이면 variables block은 다음과 같을 수 있습니다.
"""
<animal>
[a full, complete, value for the variable "animal"]
</animal>
<topic>
[a full, complete, value for the variable "topic"]
</topic>
"""


def construct_variables_block(prompt_template):
    """synthetic test data 프롬프트의 variables block을 구성합니다."""
    variables = extract_variables(prompt_template)
    output = ""
    for v in variables:
        output += f"<{v}>\n"
        output += f'[a full, complete, value for the variable "{v}". (You do not need to repeat the variable name inside the tags.)]\n'
        output += f"</{v}>\n"
    return output.strip()


# `construct_examples`는 {variable: value} dictionary를 받아 XML 형식의 예제를 만듭니다.
# 예를 들어 dict가
# {'animal': 'cat', 'topic': 'movement patterns'}이면 예제는 다음과 같습니다.
"""
<example>
<variables>
<animal>
cat
</animal>
<topic>
movement patterns
</topic>
</variables>
</example>
"""


def construct_example_block(variable_dict):
    """변수 dictionary로 example block을 구성합니다."""
    output = "<example>\n<variables>\n"
    for k, v in variable_dict.items():
        output += f"<{k}>\n{v}\n</{k}>\n"
    output = output.strip()
    output += "\n</variables>\n</example>"
    return output

## Data 생성을 위한 프롬프트 Template

이 프롬프트 template들의 일반적인 아이디어는 변수가 포함된 사용자 제출 프롬프트 template을 받아, template을 채울 변수 값을 만드는 것입니다.

아래에는 실제로 두 개의 프롬프트 template이 있습니다. 하나는 사용자가 이미 예제 변수 값을 제공했다고 가정하여 형식을 지정하고, 다른 하나는 그렇게 가정하지 않습니다.

공통점은 두 template 모두 Claude에 상황의 맥락을 제공하는 것으로 시작하며, test case를 출력하기 전에 각 변수의 사양을 개별적으로 그리고 사용자 제공 프롬프트 template 전체를 신중히 검토하도록 지시한다는 것입니다.

In [3]:
# Synthetic Evaluation용 프롬프트 Template 형식 지정

# 이 함수는 synthetic test data 생성을 위한 프롬프트 template을 준비합니다.


def format_prompt_template_for_synth_evals(prompt_template, examples=None):
    """synthetic evaluation의 프롬프트 template 형식을 지정합니다."""
    synth_test_data_prompt_template_with_example = """<Prompt Template>
{{PROMPT_TEMPLATE}}
</Prompt Template>

Your job is to construct a test case for the prompt template above. This template contains "variables", which are placeholders to be filled in later. In this case, the variables are:

<variables>
{{CONSTRUCT_VARIABLES_NAMES}}
</variables>

Here are the example test cases provided by the user.
<examples>
{{EXAMPLES}}
</examples>

First, in <planning> tags, do the following:

1. Summarize the prompt template. What is the goal of the user who created it?
2. For each variable in <variables>, carefully consider what a paradigmatic, realistic example of that variable would look like. You'll want to note who will be responsible "in prod" for supplying values. Written by a human "end user"? Downloaded from a website? Extracted from a database? Think about things like length, format, and tone in addition to semantic content. Use the examples provided by the user to guide this exercise. The goal is to acquire a sense of the statistical distribution the examples are being drawn from. The example you write should be drawn from that same distribution, but sufficiently different from the examples that it provides additional signal. A tricky balancing act, but I have faith in you.

Once you're done, output a test case for this prompt template with a full, complete, value for each variable. The output format should consist of a tagged block for each variable, with the value inside the block, like the below:

<variables>
{{CONSTRUCT_VARIABLES_BLOCK}}
</variables>"""

    synth_test_data_prompt_template_without_example = """<Prompt Template>
{{PROMPT_TEMPLATE}}
</Prompt Template>

Your job is to construct a test case for the prompt template above. This template contains "variables", which are placeholders to be filled in later. In this case, the variables are:

<variables>
{{CONSTRUCT_VARIABLES_NAMES}}
</variables>

First, in <planning> tags, do the following:

1. Summarize the prompt template. What is the goal of the user who created it?
2. For each variable in <variables>, carefully consider what a paradigmatic, realistic example of that variable would look like. You'll want to note who will be responsible "in prod" for supplying values. Written by a human "end user"? Downloaded from a website? Extracted from a database? Think about things like length, format, and tone in addition to semantic content.

Then, output a test case for this prompt template with a full, complete, value for each variable. The output format should consist of a tagged block for each variable, with the value inside the block, like the below:
<variables>
{{CONSTRUCT_VARIABLES_BLOCK}}
</variables>"""

    if examples:
        examples_block = "\n".join([construct_example_block(example) for example in examples])
        return (
            synth_test_data_prompt_template_with_example.replace(
                "{{PROMPT_TEMPLATE}}", prompt_template
            )
            .replace("{{CONSTRUCT_VARIABLES_NAMES}}", construct_variables_names(prompt_template))
            .replace("{{CONSTRUCT_VARIABLES_BLOCK}}", construct_variables_block(prompt_template))
            .replace("{{EXAMPLES}}", examples_block)
        )
    else:
        return (
            synth_test_data_prompt_template_without_example.replace(
                "{{PROMPT_TEMPLATE}}", prompt_template
            )
            .replace("{{CONSTRUCT_VARIABLES_NAMES}}", construct_variables_names(prompt_template))
            .replace("{{CONSTRUCT_VARIABLES_BLOCK}}", construct_variables_block(prompt_template))
        )

다음은 적절한 프롬프트 template을 채우고 Claude를 호출하는 또 다른 짧은 helper 함수입니다.

In [4]:
def get_test_data(prompt_template, examples, custom_planning=None):
    """Claude API로 test data를 생성합니다."""
    synth_eval_prompt_ready = format_prompt_template_for_synth_evals(prompt_template, examples)

    messages = [
        {
            "role": "user",
            "content": synth_eval_prompt_ready,
        }
    ]
    if custom_planning:
        messages.append(
            {
                "role": "assistant",
                "content": custom_planning,
            }
        )

    message = (
        CLIENT.messages.create(
            max_tokens=4000,
            messages=messages,
            model=MODEL_NAME,
            temperature=1,
        )
        .content[0]
        .text
    )

    return message

In [5]:
# 채워진 template에 대한 Claude 응답을 sampling하는 데 이 함수를 사용합니다.
# 예제 값/test case를 얻은 후에 사용합니다.


def call_claude_with_template(prompt_template, variables):
    """채워진 프롬프트 template으로 Claude를 호출합니다."""
    filled_template = prompt_template
    for var, value in variables.items():
        filled_template = filled_template.replace(f"{{{{{var}}}}}", value)

    message = (
        CLIENT.messages.create(
            max_tokens=4000,
            messages=[
                {
                    "role": "user",
                    "content": filled_template,
                }
            ],
            model=MODEL_NAME,
            temperature=0.7,
        )
        .content[0]
        .text
    )

    return message

이제 구성 요소를 결합하기 시작할 수 있습니다. 먼저 여기에 자신의 프롬프트 template을 입력하세요.

In [6]:
# 자신의 프롬프트 template으로 교체하세요!
# 변수를 나타내려면 이중 괄호 사용
# 예제:
prompt_template = """You are a customer support bot for Acme Corporation.
Here is an FAQ with Acme's relevant policies:

<documents>
{{DOCUMENTS}}
</documents>

Please respond to this customer support question using details from the policies:

<question>
{{QUESTION}}
</question>"""

variables = extract_variables(prompt_template)
print("\nIdentified variables:")
for var in variables:
    print(f"- {var}")


Identified variables:
- DOCUMENTS
- QUESTION


다음으로 입력과 이상적인 출력의 "golden example"이 있다면 입력할 수 있습니다. 현재 코드는 주석 처리되어 있습니다.

In [7]:
planning_text = None
USER_EXAMPLES = []

# if input("\nDo you want to provide an example value for your variables? (y/n): ").lower() == 'y':
#     example = {}
#     for var in variables:
#         example[var] = input(f"Enter an example value for {var}: ")
#     USER_EXAMPLES.append(example)

다음으로 이 정보로 test case 생성 프롬프트 template을 채워 test case를 얻을 수 있습니다!

In [8]:
result = get_test_data(prompt_template, USER_EXAMPLES, planning_text)

이제 test case와 Claude가 이를 생성하는 데 사용한 계획을 모두 살펴보겠습니다.

In [10]:
planning_match = re.search(r"<planning>(.*?)</planning>", result, re.DOTALL)
if planning_match and not planning_text:
    planning_text = "<planning>\n" + planning_match.group(1).strip() + "\n</planning>"

extracted_variables = {}
for var in variables:
    var_match = re.search(f"<{var}>(.*?)</{var}>", result[result.index("<variables>") :], re.DOTALL)
    if var_match:
        extracted_variables[var] = var_match.group(1).strip()

USER_EXAMPLES.append(extracted_variables)

print("~~~~~~~~~~~\nGenerated test case:\n~~~~~~~~~~~")
for var, value in extracted_variables.items():
    print(f"{var}:\n{value}\n")

print("~~~~~~~~~~~\nPlanning:\n~~~~~~~~~~~")
print(planning_text)

~~~~~~~~~~~
Generated test case:
~~~~~~~~~~~
DOCUMENTS:
Return Policy
- Items may be returned within 30 days of purchase with original receipt
- Items must be unused and in original packaging
- Shipping costs are non-refundable
- Gift cards are non-returnable

Shipping Information
- Standard shipping (5-7 business days): Free on orders over $50
- Express shipping (2-3 business days): $12.99
- Overnight shipping (next business day): $24.99
- We ship to continental US only
- Alaska and Hawaii orders incur additional $15 fee

Payment Methods
- We accept Visa, Mastercard, American Express, and PayPal
- Payment is processed at time of order
- Gift cards cannot be used for partial payment

QUESTION:
Hi, I ordered a sweater last week but it doesn't fit right. Can I return it? And will I get refunded for the shipping I paid? Thanks!

~~~~~~~~~~~
Planning:
~~~~~~~~~~~
<planning>
1. Prompt Template Summary:
This template creates a customer service chatbot for Acme Corporation that answers custom

여기서 몇 가지 방향으로 진행할 수 있습니다. test case를 더 생성하거나 Claude의 계획 로직을 수정할 수 있습니다. Claude의 계획 로직을 조금 수정해 보겠습니다. ACME 문서가 번호가 매겨진 줄을 사용한다는 것을 알고 있을 수 있습니다. 그 밖의 현실적인 변경 예시는 다음과 같습니다.

- Claude가 문서를 더 길고 상세하게 만들라고 자신에게 지시하도록 합니다.
- Claude가 고객 지원 질문의 격식을 더 높이거나 낮추라고 자신에게 지시하도록 합니다.

In [11]:
planning_text = planning_text.replace(
    "each with a question and answer format",
    "each with a question and answer format and associated number.",
)
# 계획 텍스트가 조금 다를 수 있으므로 replace를 다시 작성해야 할 수 있습니다.

예제를 초기화하되 이 계획 텍스트를 prefill로 사용하겠습니다. (sampling 시간을 조금 절약합니다.)

In [12]:
USER_EXAMPLES = []
result = get_test_data(prompt_template, USER_EXAMPLES, planning_text)

이제 새 결과를 보겠습니다.

In [13]:
# 위 셀에서 복사하여 붙여넣음.
planning_match = re.search(r"<planning>(.*?)</planning>", result, re.DOTALL)
if planning_match and not planning_text:
    planning_text = "<planning>\n" + planning_match.group(1).strip() + "\n</planning>"

extracted_variables = {}
for var in variables:
    var_match = re.search(f"<{var}>(.*?)</{var}>", result[result.index("<variables>") :], re.DOTALL)
    if var_match:
        extracted_variables[var] = var_match.group(1).strip()

USER_EXAMPLES.append(extracted_variables)

print("~~~~~~~~~~~\nGenerated test case:\n~~~~~~~~~~~")
for var, value in extracted_variables.items():
    print(f"{var}:\n{value}\n")

print("~~~~~~~~~~~\nPlanning:\n~~~~~~~~~~~")
print(planning_text)

~~~~~~~~~~~
Generated test case:
~~~~~~~~~~~
DOCUMENTS:
Return Policy
- Items may be returned within 30 days of purchase with original receipt
- Items must be unused and in original packaging
- Shipping costs are non-refundable
- Store credit will be issued for items returned without receipt

Shipping Information
- Standard shipping (5-7 business days): $5.99
- Express shipping (2-3 business days): $12.99
- Free standard shipping on orders over $50
- We currently ship only within the continental United States
- Alaska and Hawaii orders subject to additional fees

Payment Methods
- We accept Visa, Mastercard, American Express, and PayPal
- Gift cards cannot be used for online purchases
- Payment is processed at time of order
- All prices are in USD

QUESTION:
Hi, I ordered a sweater last week but it doesn't fit right. Can I return it? I still have the tags on it but I threw away the receipt. Thanks!

~~~~~~~~~~~
Planning:
~~~~~~~~~~~
<planning>
1. Prompt Template Summary:
This template 

좋습니다. 번호가 매겨진 Q와 A를 만들었습니다!

다른 예제를 만들어 보겠습니다. 이미 가진 예제를 사용하므로 흥미롭게 다른 결과가 나오기를 기대합니다.

In [14]:
result = get_test_data(prompt_template, USER_EXAMPLES, planning_text)

In [15]:
# 위 셀에서 복사하여 붙여넣음.
planning_match = re.search(r"<planning>(.*?)</planning>", result, re.DOTALL)
if planning_match and not planning_text:
    planning_text = "<planning>\n" + planning_match.group(1).strip() + "\n</planning>"

extracted_variables = {}
for var in variables:
    var_match = re.search(f"<{var}>(.*?)</{var}>", result[result.index("<variables>") :], re.DOTALL)
    if var_match:
        extracted_variables[var] = var_match.group(1).strip()

USER_EXAMPLES.append(extracted_variables)

print("~~~~~~~~~~~\nGenerated test case:\n~~~~~~~~~~~")
for var, value in extracted_variables.items():
    print(f"{var}:\n{value}\n")

print("~~~~~~~~~~~\nPlanning:\n~~~~~~~~~~~")
print(planning_text)

~~~~~~~~~~~
Generated test case:
~~~~~~~~~~~
DOCUMENTS:
Product Warranty
- All electronics come with a 1-year limited manufacturer warranty
- Warranty covers defects in materials and workmanship
- Warranty does not cover accidental damage or misuse
- Extended warranty available for purchase within 30 days

Price Match Policy
- We match prices from authorized retailers
- Item must be identical model/color/specification
- Must be in stock at competitor's store
- Online retailers excluded from price matching
- Price match requests must be made at time of purchase

Order Cancellation
- Orders can be cancelled within 2 hours of placement
- Once order is shipped, cancellation not possible
- Cancelled orders refunded to original payment method
- Processing time for refunds: 3-5 business days
- Contact customer service for cancellation requests

QUESTION:
Hello, I bought a laptop from your store 3 weeks ago and it keeps shutting down randomly. It's still under warranty, right? What do I need t

여전히 ACME corporation에 관한 것이지만 질문과 knowledge base가 모두 다릅니다.

여기서부터는 가능성이 무궁무진합니다. 코드를 loop로 실행해 test case를 더 생성하고, 계획을 더 수정하고, 이 test case에서 Claude를 평가하고, 만든 test case와 golden answer를 multishot 예제로 프롬프트에 넣을 수 있습니다.

golden answer를 얻으려면 처음부터 직접 작성하거나 Claude가 답변을 작성하게 한 뒤 원하는 대로 편집할 수 있습니다. prompt caching이 등장한 지금은 성능 개선을 위해 프롬프트에 많은 예제를 추가하기에 그 어느 때보다 좋은 시점입니다.